# CodeReviewAgent — GRPO Training Notebook

Fine-tunes a small LLM to perform structured code reviews using **Group Relative Policy Optimization (GRPO)** via HuggingFace TRL.

**Environment**: OpenEnv CodeReviewAgent — 5 Python code tasks of increasing difficulty (easy → hard).  
**Task**: Given a code snippet, output a JSON array of all bugs, security issues, and design problems.  
**Reward**: Keyword + line-range matching against ground-truth issues.

**Runtime**: T4 GPU (free Colab tier) — ~25-40 min for 3 epochs on 1.5B model.

## 1. Install Dependencies

In [ ]:
# Install Unsloth for fast LoRA fine-tuning, then TRL + OpenEnv
!pip install -q unsloth
!pip install -q trl>=0.12 datasets accelerate
!pip install -q openenv

# Clone the environment repo (or install from HF Hub)
!git clone --depth 1 https://huggingface.co/spaces/themahipalthakur/CodeReviewAgent /content/CodeReviewAgent 2>/dev/null || echo 'Already cloned'
%cd /content/CodeReviewAgent
!pip install -q -e .

## 2. Imports & Config

In [ ]:
import json
import re
import os
import sys
from typing import Any

import torch
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

sys.path.insert(0, '/content/CodeReviewAgent')
from CodeReviewAgent.server.tasks import TASKS

MODEL_ID       = 'unsloth/Qwen2.5-1.5B-Instruct'  # change to 3B for better results
OUTPUT_DIR     = '/content/grpo_output'
NUM_EPOCHS     = 3
NUM_GENERATIONS = 4   # completions sampled per prompt
MAX_NEW_TOKENS  = 600
REPEAT_DATASET  = 8   # repeats per task for more training steps

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Tasks: {len(TASKS)}')
for t in TASKS:
    print(f'  Task {t["id"]} [{t["difficulty"]:>6}]: {t["name"]} ({len(t["issues"])} issues)')

## 3. Build Training Dataset

In [ ]:
SYSTEM_PROMPT = """\
You are an expert code reviewer. Analyze the provided Python code carefully and \
identify ALL issues including bugs, security vulnerabilities, performance problems, \
and design issues.

Output your review as a JSON array. Each element must be an object with these fields:
  \"line\"     : integer — the primary line number of the issue
  \"category\" : one of \"bug\", \"security\", \"performance\", \"style\", \"design\"
  \"severity\" : one of \"info\", \"warning\", \"error\", \"critical\"
  \"comment\"  : string — clear description of the problem and how to fix it

Output ONLY valid JSON (no markdown fences, no extra text).\
"""

def make_prompt(task):
    return (
        f"File: {task['file_name']}\n"
        f"Task: {task['description']}\n\n"
        f"```python\n{task['code']}\n```\n\n"
        "Provide your review as a JSON array of issues:"
    )

rows = []
for task in TASKS:
    rows.append({
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': make_prompt(task)},
        ],
        'task_id': task['id'],
    })

dataset = Dataset.from_list(rows * REPEAT_DATASET)
print(f'Dataset: {len(dataset)} rows ({len(TASKS)} tasks × {REPEAT_DATASET} repeats)')

## 4. Reward Function

In [ ]:
def _parse_json(text: str) -> list[dict]:
    text = text.strip()
    fence = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if fence:
        text = fence.group(1).strip()
    parsed = json.loads(text)
    if not isinstance(parsed, list):
        raise ValueError('not a list')
    return parsed

def score_review(completion: str, task: dict) -> float:
    try:
        issues = _parse_json(completion)
    except Exception:
        return -0.10

    total_w = sum(i['weight'] for i in task['issues'])
    found_ids, false_pos = set(), 0

    for found in issues:
        if not isinstance(found, dict):
            continue
        comment  = str(found.get('comment', '')).lower()
        line     = found.get('line')
        category = found.get('category')
        matched  = False

        for gt in task['issues']:
            if gt['id'] in found_ids:
                continue
            kw_hit  = any(kw.lower() in comment for kw in gt['keywords'])
            s, e    = gt['line_range']
            line_hit = line is not None and (s - 3) <= int(line) <= (e + 3)
            cat_hit  = category == gt['category']
            if kw_hit and (line_hit or cat_hit):
                found_ids.add(gt['id'])
                matched = True
                break
        if not matched and len(comment) > 15:
            false_pos += 1

    found_w  = sum(gt['weight'] for gt in task['issues'] if gt['id'] in found_ids)
    coverage = found_w / total_w if total_w else 0.0
    reward   = coverage * 0.70 - false_pos * 0.02
    return float(max(-0.20, min(1.0, reward)))

def reward_fn(completions: list[str], task_id: list[int], **_) -> list[float]:
    return [score_review(c, TASKS[tid % len(TASKS)]) for c, tid in zip(completions, task_id)]

# Smoke test
perfect = json.dumps([
    {'line': 4,  'category': 'bug',   'severity': 'error', 'comment': 'Off-by-one error in range causes IndexError'},
    {'line': 7,  'category': 'style', 'severity': 'info',  'comment': 'unused_result variable is never used'},
    {'line': 17, 'category': 'bug',   'severity': 'error', 'comment': 'max_val == item uses == not = so max is never updated'},
])
r = score_review(perfect, TASKS[0])
print(f'Perfect review on task-0 reward: {r:.4f}  (expect ≈ 0.70)')
print(f'Empty array reward:              {score_review("[]", TASKS[0]):.4f}  (expect 0.0)')
print(f'Malformed JSON reward:           {score_review("bad", TASKS[0]):.4f}  (expect -0.10)')

## 5. Load Model (Unsloth + LoRA)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)
print('Model loaded with LoRA adapters.')

## 6. Train with GRPO

In [ ]:
config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_new_tokens=MAX_NEW_TOKENS,
    max_prompt_length=1024,
    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=1,
    save_strategy='epoch',
    report_to='none',
    bf16=True,
    temperature=0.9,
    top_p=0.95,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=[reward_fn],
)

trainer.train()

## 7. Save Model

In [ ]:
final_path = os.path.join(OUTPUT_DIR, 'final')
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f'Model saved to {final_path}')

## 8. Evaluate — Before vs After

In [ ]:
import statistics

def evaluate_model(model, tokenizer, n_samples=3) -> dict:
    """Run the trained model on all tasks and report reward per task."""
    results = {}
    model.eval()
    FastLanguageModel.for_inference(model)

    for task in TASKS:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': make_prompt(task)},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)

        rewards = []
        for _ in range(n_samples):
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            rewards.append(score_review(gen, task))

        results[task['id']] = {
            'difficulty': task['difficulty'],
            'name': task['name'],
            'mean_reward': statistics.mean(rewards),
            'rewards': rewards,
        }
    return results

print('Evaluating trained model …')
eval_results = evaluate_model(model, tokenizer)
print(f'\n{"Task":<6} {"Difficulty":<10} {"Mean Reward":>12}  Name')
print('─' * 55)
for tid, r in eval_results.items():
    print(f'{tid:<6} {r["difficulty"]:<10} {r["mean_reward"]:>12.4f}  {r["name"]}')
avg = statistics.mean(r['mean_reward'] for r in eval_results.values())
print(f'\nOverall mean reward: {avg:.4f}')

## 9. Plot Reward Curves

In [ ]:
import json
import glob
import matplotlib.pyplot as plt
import numpy as np

# Load training logs
log_file = os.path.join(OUTPUT_DIR, 'trainer_state.json')

if os.path.exists(log_file):
    with open(log_file) as f:
        state = json.load(f)

    log_history = state.get('log_history', [])
    steps, rewards, losses = [], [], []
    for entry in log_history:
        if 'reward' in entry:
            steps.append(entry.get('step', len(steps)))
            rewards.append(entry['reward'])
        if 'loss' in entry:
            losses.append(entry['loss'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Reward curve with smoothing
    ax = axes[0]
    ax.plot(steps, rewards, alpha=0.3, color='steelblue', label='raw')
    if len(rewards) >= 5:
        smooth = np.convolve(rewards, np.ones(5)/5, mode='valid')
        ax.plot(steps[2:2+len(smooth)], smooth, color='steelblue', linewidth=2, label='smoothed (5-step)')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Reward')
    ax.set_title('GRPO Reward During Training')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Loss curve
    ax = axes[1]
    if losses:
        ax.plot(losses, color='coral')
        ax.set_xlabel('Training step')
        ax.set_ylabel('Loss')
        ax.set_title('Training Loss')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved to /content/training_curves.png')
else:
    print(f'No log file found at {log_file} — run training first.')

# Per-task reward bar chart
fig, ax = plt.subplots(figsize=(9, 5))
tids   = list(eval_results.keys())
names  = [f"Task {t}\n{eval_results[t]['difficulty']}" for t in tids]
scores = [eval_results[t]['mean_reward'] for t in tids]
colors = ['#4CAF50' if s > 0.4 else '#FF9800' if s > 0.1 else '#F44336' for s in scores]
ax.bar(names, scores, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Reward')
ax.set_title('Post-Training Mean Reward per Task')
ax.set_ylim(-0.25, 1.0)
for i, s in enumerate(scores):
    ax.text(i, s + 0.02, f'{s:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/content/per_task_reward.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to /content/per_task_reward.png')